# Global LULUCF + vegetation zonal stats by IPCC land use

This notebook reads global 10×10 tile-level zonal stats parquet files one at a time and summarizes them.

Outputs use sums only:

- `value` = sum of flux values
- `area_ha` = sum of area

The workflow is:
1. Read one tile parquet.
2. Summarize by strata.
3. Write compact tile summary to disk.
4. Delete raw tile data from memory.
5. After all tile summaries exist, combine compact summaries into one global master table.
6. Export Excel, CSV, and parquet.

In [1]:
from pathlib import Path
import gc
import time
from datetime import datetime

import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:,.3f}".format)

In [2]:
# Update these paths if needed.
input_dir = Path(r"/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global/tile_stats")
output_dir = Path(r"/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global")
file_keyword = "iso"

tile_summary_dir = output_dir / f"tile_summaries__{file_keyword}"
tile_summary_dir.mkdir(parents=True, exist_ok=True)

print("Input folder:", input_dir)
print("Output folder:", output_dir)
print("Tile summary folder:", tile_summary_dir)

ipcc_csv_path = Path(r"/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global/rules/ipcc_rules.csv")
print("IPCC rules:", ipcc_csv_path)


Input folder: /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global/tile_stats
Output folder: /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global
Tile summary folder: /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global/tile_summaries__iso
IPCC rules: /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global/rules/ipcc_rules.csv


In [3]:
numeric_to_ipcc_class = {
    0: "Unassigned",
    1: "Settlements",
    2: "Cropland",
    3: "Forest Land",
    4: "Grassland",
    5: "Wetlands",
    6: "Other Land (Bare)",
    7: "Other Land (Water)",
    8: "Other Land (Snow/Ice)",
}

assignment_rules = [
    {
        "name": "crop_to_cropland",
        "match": {"land_state_broad_class": "crop"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "short_veg_to_grassland",
        "match": {"land_state_broad_class": "short_veg"},
        "fill": {"IPCC_summary": 44, "IPCC_change": 44, "IPCC_class": 4, "IPCC_class_name": "Grassland"},
    },
    {
        "name": "mangrove_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "mangrove"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "natural_tree_cover_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "natural_tree_cover"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "oil_palm_to_cropland",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "oil_palm"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "non_oil_palm_planted_trees_to_cropland",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "non_oil_palm_planted_trees"},
        "fill": {"IPCC_summary": 22, "IPCC_change": 22, "IPCC_class": 2, "IPCC_class_name": "Cropland"},
    },
    {
        "name": "trees_in_other_land_covers_to_forest",
        "match": {"land_state_broad_class": "tree", "tall_veg_type": "trees_in_other_land_covers"},
        "fill": {"IPCC_summary": 33, "IPCC_change": 33, "IPCC_class": 3, "IPCC_class_name": "Forest Land"},
    },
    {
        "name": "no_flux_stays_unassigned",
        "match": {"land_state_broad_class": "no_flux"},
        "fill": {"IPCC_summary": 0, "IPCC_change": 0, "IPCC_class": 0, "IPCC_class_name": "Unassigned"},
    },
]

In [4]:
def log(message):
    print(f"{time.strftime('%Y%m%d_%H_%M_%S')}: {message}", flush=True)

def find_tile_parquets(input_dir, file_keyword):
    return sorted(
        p for p in input_dir.glob("*.parquet")
        if "ipcc_lulucf_zonal_stats_" in p.name
        and file_keyword in p.name
        and "_wide_" not in p.name
        and "summary" not in p.name
        and "combined" not in p.name
    )

def tile_id_from_parquet_name(parquet_file):
    parts = parquet_file.stem.split("_")
    if len(parts) >= 6:
        return f"{parts[4]}_{parts[5]}"
    return parquet_file.stem

def coalesce_node_code_column(df):
    if "IPCC_node_code" in df.columns:
        return df

    if "IPCC_node" in df.columns:
        return df.rename(columns={"IPCC_node": "IPCC_node_code"})

    return df

def coalesce_primary_forest_column(df):
    # Keep the requested grouping column name stable even if the source has a slightly different name.
    possible_cols = [
        "starting_composite_primary_forest",
        "starting_composite_primary_forest_2001",
        "composite_primary_forest",
    ]

    if "starting_composite_primary_forest" in df.columns:
        return df

    for col in possible_cols:
        if col in df.columns:
            return df.rename(columns={col: "starting_composite_primary_forest"})

    # If the source does not contain the column, keep a sentinel so the notebook still runs.
    df["starting_composite_primary_forest"] = "Unassigned"
    return df


def add_flux_source(df):
    df["flux_source"] = df["analysis_layer"].apply(
        lambda x: "LULUCF" if str(x).startswith("LULUCF_") else "vegetation"
    )
    return df

LAYERS_TO_KEEP = [
    # LULUCF layers
    "LULUCF_gross_emissions__all_C_pools__all_gases__MgCO2e",
    "LULUCF_gross_removals__all_C_pools__MgCO2",
    "LULUCF_net_flux__all_C_pools__all_gases__MgCO2e",

    # Vegetation layers, if present
    "gross_emissions__all_C_pools__all_gases__MgCO2e",
    "gross_removals__all_C_pools__MgCO2",
    "net_flux__all_C_pools__all_gases__MgCO2e",
]


SUMMARY_COLS = [
    "flux_source",
    "adm0",
    "IPCC_summary",
    "IPCC_node_code",
    "year",
    "IPCC_change",
    "IPCC_class",
    "IPCC_class_name",
    "starting_composite_primary_forest",
    "land_state_broad_class",
    "tall_veg_type",
    "land_state_detailed_class",
    "land_state_meaning",
    "analysis_layer",
]

VALUE_COLS = ["value", "area_ha"]

OUTPUT_COLS = SUMMARY_COLS + VALUE_COLS

def apply_assignment_rule(df, rules=assignment_rules):
    if "IPCC_class_name" not in df.columns:
        return df

    df["IPCC_class_name"] = df["IPCC_class_name"].fillna("Unassigned")
    base_unassigned = df["IPCC_class_name"].astype(str).eq("Unassigned")

    for rule in rules:
        mask = base_unassigned.copy()

        for col, val in rule["match"].items():
            if col not in df.columns:
                mask &= False
            else:
                mask &= df[col].astype(str).eq(str(val))

        if mask.any():
            for fill_col, fill_val in rule["fill"].items():
                df.loc[mask, fill_col] = fill_val

    for col in ["IPCC_summary", "IPCC_change", "IPCC_class"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")

    class_map_mask = df["IPCC_class_name"].isna() | df["IPCC_class_name"].astype(str).eq("")
    if "IPCC_class" in df.columns:
        df.loc[class_map_mask, "IPCC_class_name"] = df.loc[class_map_mask, "IPCC_class"].map(numeric_to_ipcc_class).fillna("Unassigned")

    return df

IPCC_KEY_COLS = [
    "IPCC_summary",
    "IPCC_node_code",
    "IPCC_change",
    "IPCC_class",
    "IPCC_class_name",
    "starting_composite_primary_forest",
    "land_state_broad_class",
    "tall_veg_type",
    "land_state_detailed_class",
    "land_state_meaning",
]

IPCC_OUTPUT_COLS = [
    "IPCC_summary",
    "IPCC_node_code",
    "IPCC_change",
    "IPCC_class",
    "IPCC_class_name",
]


def load_ipcc_rules(rules_path):
    if not rules_path.exists():
        raise FileNotFoundError(
            f"IPCC rules not found: {rules_path} ipcc_csv_path in the configuration cell."
        )

    rules = pd.read_csv(rules_path, keep_default_na=True)
    updated_cols = [f"{col}_updated" for col in IPCC_OUTPUT_COLS]
    required_cols = IPCC_KEY_COLS + updated_cols
    missing = [col for col in required_cols if col not in rules.columns]

    if missing:
        raise ValueError(f"Rules file is missing required columns: {missing}")

    rules = rules[required_cols].drop_duplicates().reset_index(drop=True)

    # Each original condition must map to exactly one corrected result.
    conflicts = (
        rules.groupby(IPCC_KEY_COLS, dropna=False)[updated_cols]
             .nunique(dropna=False)
             .max(axis=1)
             .gt(1)
    )
    if conflicts.any():
        raise ValueError(
            f"The rules file contains {int(conflicts.sum()):,} conflicting conditions."
        )

    return rules


def apply_ipcc_rules(master_df, rules):
    missing = [col for col in IPCC_KEY_COLS if col not in master_df.columns]
    if missing:
        raise ValueError(f"Summary is missing rule-key columns: {missing}")

    original_columns = master_df.columns.tolist()
    updated_cols = [f"{col}_updated" for col in IPCC_OUTPUT_COLS]

    corrected = master_df.copy()
    corrected["_original_row_order"] = range(len(corrected))

    corrected = corrected.merge(
        rules,
        how="left",
        on=IPCC_KEY_COLS,
        validate="many_to_one",
        indicator="_ipcc_rule_match",
        sort=False,
    )

    matched = corrected["_ipcc_rule_match"].eq("both")
    log(
        f"Applied IPCC reassignment rules to {int(matched.sum()):,} "
        f"of {len(corrected):,} combined-summary rows"
    )

    for col in IPCC_OUTPUT_COLS:
        corrected.loc[matched, col] = corrected.loc[matched, f"{col}_updated"]

    corrected = (
        corrected.sort_values("_original_row_order")
                 .drop(columns=updated_cols + ["_ipcc_rule_match", "_original_row_order"])
                 .reset_index(drop=True)
    )
    corrected = corrected[original_columns]

    for col in ["IPCC_summary", "IPCC_node_code", "year", "IPCC_change", "IPCC_class"]:
        if col in corrected.columns:
            corrected[col] = pd.to_numeric(corrected[col], errors="raise").astype("int64")

    if len(corrected) != len(master_df):
        raise ValueError("Applying IPCC rules unexpectedly changed the row count.")

    return corrected


In [16]:
def summarize_one_tile(parquet_file, tile_summary_dir):
    tile_id = tile_id_from_parquet_name(parquet_file)
    out_path = tile_summary_dir / f"{tile_id}__summary.parquet"

    if out_path.exists():
        log(f"Skipping {tile_id}; tile summary already exists")
        return out_path

    log(f"Reading {tile_id}: {parquet_file.name}")

    df = pd.read_parquet(parquet_file)
    df = coalesce_node_code_column(df)
    df = coalesce_primary_forest_column(df)

    if "analysis_layer" not in df.columns:
        raise ValueError(f"analysis_layer missing from {parquet_file.name}")

    df = add_flux_source(df)
    df = df[df["analysis_layer"].isin(LAYERS_TO_KEEP)].copy()

    if df.empty:
        log(f"No matching flux layers found in {tile_id}. Writing empty summary.")
        empty = pd.DataFrame(columns=OUTPUT_COLS)
        empty.to_parquet(out_path, index=False)
        del df, empty
        gc.collect()
        return out_path

    df = apply_assignment_rule(df)

    missing_cols = [c for c in SUMMARY_COLS + VALUE_COLS if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Missing expected columns in {parquet_file.name}: {missing_cols}\n"
            f"Available columns: {df.columns.tolist()}"
        )

    summary = (
        df.groupby(SUMMARY_COLS, dropna=False, as_index=False)
          .agg({
              "value": "sum",
              "area_ha": "sum",
          })
    )

    summary = summary[OUTPUT_COLS].sort_values(SUMMARY_COLS).reset_index(drop=True)
    summary.to_parquet(out_path, index=False)

    log(f"Wrote {tile_id} summary: {len(summary):,} rows -> {out_path.name}")

    del df, summary
    gc.collect()

    return out_path


def combine_tile_summaries(tile_summary_dir):
    summary_files = sorted(tile_summary_dir.glob("*.parquet"))

    if not summary_files:
        raise FileNotFoundError(
            f"No tile summary Parquet files found in {tile_summary_dir}"
        )

    parts = []

    for i, summary_file in enumerate(summary_files, start=1):
        part = pd.read_parquet(summary_file)

        # Skip completely empty files.
        if part.empty:
            print(f"Skipping empty summary: {summary_file.name}")
            continue

        parts.append(part)

        if i % 25 == 0 or i == len(summary_files):
            rows_loaded = sum(len(df) for df in parts)

            print(
                f"{datetime.now():%Y%m%d_%H_%M_%S}: "
                f"Loaded {i}/{len(summary_files)}; "
                f"rows loaded={rows_loaded:,}"
            )

    if not parts:
        raise ValueError(
            f"All tile summary files in {tile_summary_dir} were empty."
        )

    master = pd.concat(
        parts,
        ignore_index=True,
        sort=False,
    )

    return master


def export_summary(
    master_df,
    output_dir,
    output_stem="global_reassignment_value_area_primary_summary",
):
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H_%M_%S")

    excel_out = output_dir / f"{output_stem}__{timestamp}.xlsx"
    csv_out = output_dir / f"{output_stem}__{timestamp}.csv"
    parquet_out = output_dir / f"{output_stem}__{timestamp}.parquet"

    # Always write parquet
    master_df.to_parquet(
        parquet_out,
        index=False,
        engine="pyarrow",
    )

    print(f"Parquet written: {parquet_out}")

    # Excel maximum number of rows
    EXCEL_MAX_ROWS = 1_048_576

    if len(master_df) <= EXCEL_MAX_ROWS:
        # CSV
        master_df.to_csv(
            csv_out,
            index=False,
            float_format="%.3f",
        )

        # Excel
        with pd.ExcelWriter(excel_out, engine="xlsxwriter") as writer:
            sheet_name = "LULUCF_flux_master"
            master_df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
            )

            workbook = writer.book
            worksheet = writer.sheets[sheet_name]

            worksheet.freeze_panes(1, 0)

            number_format = workbook.add_format({"num_format": "#,##0.000"})

            for col in ["value", "area_ha"]:
                if col in master_df.columns:
                    idx = master_df.columns.get_loc(col)
                    worksheet.set_column(idx, idx, 18, number_format)

        print(f"CSV written: {csv_out}")
        print(f"Excel written: {excel_out}")

    else:
        print(
            f"Dataset contains {len(master_df):,} rows, exceeding Excel's "
            f"limit of {EXCEL_MAX_ROWS:,} rows."
        )
        print("Skipping CSV and Excel export; Parquet has been written.")

    return excel_out, csv_out, parquet_out

## Find tile parquet files

In [6]:
parquet_files = find_tile_parquets(input_dir, "global")
print("Tile parquet count:", len(parquet_files))

Tile parquet count: 356


## Inspect one tile before running all


In [7]:
if parquet_files:
    sample = pd.read_parquet(parquet_files[0])

    print("column values:")
    print(sample.columns.tolist())

    print("analysis_layer values:")
    print(sorted(sample["analysis_layer"].dropna().unique().tolist()))

    del sample
    gc.collect()

column values:
['analysis_layer', 'adm0', 'land_state_node', 'cont_eco', 'starting_composite_primary_forest', 'drivers_of_TCL_1_km', 'IPCC_class', 'IPCC_node_code', 'IPCC_change', 'IPCC_summary', 'year', 'value', 'tile_id', 'area_ha', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type', 'gas', 'country_name', 'region_L1', 'region_L2_L3', 'continent', 'ecozone', 'continent_ecozone', 'climate_domain', 'driver_1km_text', 'IPCC_class_name', 'IPCC_change_name', 'IPCC_node_code_name', 'IPCC_summary_name', 'density__Mg_ha']
analysis_layer values:
['LULUCF_gross_emissions__all_C_pools__all_gases__MgCO2e', 'LULUCF_gross_removals__all_C_pools__MgCO2', 'LULUCF_net_flux__all_C_pools__all_gases__MgCO2e', 'net_flux__all_C_pools__all_gases__MgCO2e']


## Step 1: summarize each tile


In [8]:
for idx, parquet_file in enumerate(parquet_files, start=1):
    log(f"Tile {idx:,}/{len(parquet_files):,}")
    summarize_one_tile(parquet_file, tile_summary_dir)
    gc.collect()

20260731_15_42_39: Tile 1/356
20260731_15_42_39: Skipping 00N_000E; tile summary already exists
20260731_15_42_39: Tile 2/356
20260731_15_42_39: Skipping 00N_010E; tile summary already exists
20260731_15_42_39: Tile 3/356
20260731_15_42_39: Skipping 00N_020E; tile summary already exists
20260731_15_42_39: Tile 4/356
20260731_15_42_39: Skipping 00N_020W; tile summary already exists
20260731_15_42_39: Tile 5/356
20260731_15_42_39: Skipping 00N_030E; tile summary already exists
20260731_15_42_39: Tile 6/356
20260731_15_42_39: Skipping 00N_040E; tile summary already exists
20260731_15_42_39: Tile 7/356
20260731_15_42_39: Skipping 00N_040W; tile summary already exists
20260731_15_42_39: Tile 8/356
20260731_15_42_39: Skipping 00N_050E; tile summary already exists
20260731_15_42_39: Tile 9/356
20260731_15_42_39: Skipping 00N_050W; tile summary already exists
20260731_15_42_39: Tile 10/356
20260731_15_42_39: Skipping 00N_060W; tile summary already exists
20260731_15_42_39: Tile 11/356
20260731

## Phase 2: combine compact tile summaries

### Combine tile summaries and apply IPCC rules

In [9]:
ipcc_rules = load_ipcc_rules(ipcc_csv_path)
print(f"Loaded IPCC rules")

master_df = combine_tile_summaries(tile_summary_dir)
master_df = apply_ipcc_rules(master_df, ipcc_rules)

print(master_df.shape)
master_df.head(25)


Loaded IPCC rules
Skipping empty summary: 00N_020W__summary.parquet
Skipping empty summary: 00N_140W__summary.parquet
Skipping empty summary: 00N_150W__summary.parquet
20260731_15_43_01: Loaded 25/356; rows loaded=1,444,614
Skipping empty summary: 00N_160W__summary.parquet
Skipping empty summary: 00N_170W__summary.parquet
20260731_15_43_04: Loaded 50/356; rows loaded=3,594,827
Skipping empty summary: 10N_140E__summary.parquet
Skipping empty summary: 10N_160W__summary.parquet
Skipping empty summary: 10N_170W__summary.parquet
Skipping empty summary: 10N_180W__summary.parquet
Skipping empty summary: 10S_010W__summary.parquet
Skipping empty summary: 10S_060E__summary.parquet
Skipping empty summary: 10S_090E__summary.parquet
Skipping empty summary: 10S_100E__summary.parquet
20260731_15_43_06: Loaded 75/356; rows loaded=4,412,799
Skipping empty summary: 10S_140W__summary.parquet
Skipping empty summary: 10S_170W__summary.parquet
20260731_15_43_08: Loaded 100/356; rows loaded=5,992,496
Skippin

,flux_source,adm0,IPCC_summary,IPCC_node_code,year,IPCC_change,IPCC_class,IPCC_class_name,starting_composite_primary_forest,land_state_broad_class,tall_veg_type,land_state_detailed_class,land_state_meaning,analysis_layer,value,area_ha
0,LULUCF,GAB,66,0,2016,66,6,Other Land (Bare),0,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_gross_emissions__all_C_pools__all_gases...,42.375,13.692
1,LULUCF,GAB,66,0,2016,66,6,Other Land (Bare),0,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_gross_removals__all_C_pools__MgCO2,-5.026,13.692
2,LULUCF,GAB,66,0,2016,66,6,Other Land (Bare),0,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_net_flux__all_C_pools__all_gases__MgCO2e,37.349,13.692
3,LULUCF,GAB,66,0,2016,66,6,Other Land (Bare),1,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_gross_emissions__all_C_pools__all_gases...,4.231,1.154
4,LULUCF,GAB,66,0,2016,66,6,Other Land (Bare),1,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_gross_removals__all_C_pools__MgCO2,-0.338,1.462
5,LULUCF,GAB,66,0,2016,66,6,Other Land (Bare),1,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_net_flux__all_C_pools__all_gases__MgCO2e,3.892,1.538
6,LULUCF,GAB,66,0,2017,66,6,Other Land (Bare),0,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_gross_emissions__all_C_pools__all_gases...,41.884,13.154
7,LULUCF,GAB,66,0,2017,66,6,Other Land (Bare),0,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_gross_removals__all_C_pools__MgCO2,-4.637,13.154
8,LULUCF,GAB,66,0,2017,66,6,Other Land (Bare),0,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_net_flux__all_C_pools__all_gases__MgCO2e,37.247,13.154
9,LULUCF,GAB,66,0,2017,66,6,Other Land (Bare),1,no_flux,non_tall_vegetation,no_flux,Not in decision tree,LULUCF_gross_emissions__all_C_pools__all_gases...,4.231,1.154


## QA

In [10]:
qa = (
    master_df.groupby(["flux_source", "analysis_layer"], dropna=False)[["value", "area_ha"]]
             .sum()
             .reset_index()
             .sort_values(["flux_source", "analysis_layer"])
)

qa

,flux_source,analysis_layer,value,area_ha
0,LULUCF,LULUCF_gross_emissions__all_C_pools__all_gases...,"183,609,442,304.000","141,216,923,648.000"
1,LULUCF,LULUCF_gross_removals__all_C_pools__MgCO2,"-215,606,263,808.000","140,561,809,408.000"
2,LULUCF,LULUCF_net_flux__all_C_pools__all_gases__MgCO2e,"-31,996,813,312.000","141,333,610,496.000"
3,vegetation,net_flux__all_C_pools__all_gases__MgCO2e,"-66,677,743,616.000","43,585,536,000.000"


In [11]:
master_df.groupby(["IPCC_class", "IPCC_class_name"], dropna=False)[["value", "area_ha"]].sum().reset_index()

,IPCC_class,IPCC_class_name,value,area_ha
0,1,Settlements,"-21,919,889,408.000","14,110,001,152.000"
1,2,Cropland,"31,634,987,008.000","37,633,642,496.000"
2,3,Forest Land,"-166,973,489,152.000","160,096,894,976.000"
3,4,Grassland,"26,685,739,008.000","111,038,971,904.000"
4,5,Wetlands,"-1,655,922,432.000","77,638,107,136.000"
5,6,Other Land (Bare),"1,564,827,392.000","65,271,771,136.000"
6,6,Other Land (Snow/Ice),"-7,614,333.500","908,477,504.000"


## Phase 3: export Excel, CSV, and parquet

In [17]:
excel_out, csv_out, parquet_out = export_summary(master_df, output_dir)

print("Excel:", excel_out)
print("CSV:", csv_out)
print("Parquet:", parquet_out)

Parquet written: /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global/global_reassignment_value_area_primary_summary__20260731_15_52_07.parquet
Dataset contains 13,513,162 rows, exceeding Excel's limit of 1,048,576 rows.
Skipping CSV and Excel export; Parquet has been written.
Excel: /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global/global_reassignment_value_area_primary_summary__20260731_15_52_07.xlsx
CSV: /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global/global_reassignment_value_area_primary_summary__20260731_15_52_07.csv
Parquet: /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/IPCC_land_use_v1_0_0_standard_global/global_reassignment_value_area_primary_summary__20260731_15_52_07.parquet


## Optional QA checks

In [18]:
print("Rows:", len(master_df))
print("Total value:", master_df["value"].sum())
print("Total area_ha:", master_df["area_ha"].sum())

Rows: 13513162
Total value: -130671410000.0
Total area_ha: 466697850000.0


In [19]:
qa_failures = {
    "IPCC_summary == 0": int((master_df["IPCC_summary"] == 0).sum()),
    "IPCC_change == 0": int((master_df["IPCC_change"] == 0).sum()),
    "IPCC_class == 0": int((master_df["IPCC_class"] == 0).sum()),
    "IPCC_class_name == 'Unassigned'": int(
        master_df["IPCC_class_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("Unassigned")
        .sum()
    ),
}

remaining_failures = {
    check: count
    for check, count in qa_failures.items()
    if count > 0
}

if remaining_failures:
    failure_text = ", ".join(
        f"{check}: {count:,} rows"
        for check, count in remaining_failures.items()
    )

    raise AssertionError(
        "IPCC reassignment QA failed. "
        "The following unassigned values remain:\n"
        + failure_text
    )

print("✓ IPCC reassignment QA passed: no zero or 'Unassigned' IPCC values remain.")

✓ IPCC reassignment QA passed: no zero or 'Unassigned' IPCC values remain.


In [20]:
master_df.groupby("year", dropna=False)[["value", "area_ha"]].sum()

,value,area_ha
year,,
2016,"-13,930,273,792.000","51,828,785,152.000"
2017,"-16,496,849,920.000","51,893,600,256.000"
2018,"-16,535,686,144.000","51,861,618,688.000"
2019,"-14,638,320,640.000","51,840,151,552.000"
2020,"-10,685,446,144.000","51,850,559,488.000"
2021,"-13,873,137,664.000","51,850,620,928.000"
2022,"-15,988,365,312.000","51,822,764,032.000"
2023,"-14,625,637,376.000","51,868,450,816.000"
2024,"-13,897,652,224.000","51,881,324,544.000"
